***1. Embedded Method using L1 Regularization (Lasso)***

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv('loan_default.csv')

# Remove unnecessary column
df = df.drop(columns='LoanID')

# Remove duplicate columns
#df = df.loc[:, ~df.T.duplicated()]

df = df.dropna(subset=['Default'])

# Separate features and target
X = df.drop('Default', axis=1)
y = df['Default']

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert back to DataFrame
X_train = pd.DataFrame(X_train, columns=X.columns)
X_test = pd.DataFrame(X_test, columns=X.columns)

# Logistic Regression with L1 Regularization
lr_l1 = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=0.1
)

# Train model
lr_l1.fit(X_train, y_train)

# Prediction
y_pred = lr_l1.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("L1 Accuracy:", accuracy)

# Feature coefficients
coef = pd.Series(
    lr_l1.coef_[0],
    index=X_train.columns
)

# Selected features
selected_features = coef[coef != 0]

print("\nSelected Features:")
print(selected_features)

print("\nTotal Selected Features:", len(selected_features))

L1 Accuracy: 0.8852163696886626

Selected Features:
Age                            -0.583932
Income                         -0.341097
LoanAmount                      0.299520
CreditScore                    -0.121352
MonthsEmployed                 -0.336250
NumCreditLines                  0.099783
InterestRate                    0.457374
LoanTerm                        0.003318
DTIRatio                        0.067688
Education_High School           0.032740
Education_Master's             -0.056907
Education_PhD                  -0.076573
EmploymentType_Part-time        0.117376
EmploymentType_Self-employed    0.104315
EmploymentType_Unemployed       0.190992
MaritalStatus_Married          -0.097518
MaritalStatus_Single           -0.027864
HasMortgage_Yes                -0.076969
HasDependents_Yes              -0.125696
LoanPurpose_Business            0.022944
LoanPurpose_Education          -0.002628
LoanPurpose_Home               -0.072663
LoanPurpose_Other              -0.003063
HasCo

***2. Embedded Method using Tree-Based Feature Importance***

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv('loan_default.csv')

# Remove unnecessary column
df = df.drop(columns='LoanID')

# Remove duplicate columns
#df = df.loc[:, ~df.T.duplicated()]

df = df.dropna(subset=['Default'])

# Separate features and target
X = df.drop('Default', axis=1)
y = df['Default']

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Random Forest model
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train model
rf.fit(X_train, y_train)

# Prediction
y_pred = rf.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Random Forest Accuracy:", accuracy)

# Feature importance
importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
)

# Sort importance
importance = importance.sort_values(ascending=False)

print("\nFeature Importance:")
print(importance)

# Select important features
selected_features = importance[importance > 0.01]

print("\nSelected Features:")
print(selected_features)

print("\nTotal Selected Features:", len(selected_features))

Random Forest Accuracy: 0.8853338554924614

Feature Importance:
Income                          0.126087
InterestRate                    0.120195
LoanAmount                      0.113021
Age                             0.102290
CreditScore                     0.100056
MonthsEmployed                  0.097823
DTIRatio                        0.088182
LoanTerm                        0.040419
NumCreditLines                  0.031977
HasMortgage_Yes                 0.014890
MaritalStatus_Single            0.014124
EmploymentType_Part-time        0.012308
MaritalStatus_Married           0.012289
EmploymentType_Self-employed    0.012144
HasDependents_Yes               0.012091
LoanPurpose_Business            0.011923
LoanPurpose_Education           0.011855
Education_Master's              0.011843
LoanPurpose_Other               0.011839
Education_High School           0.011733
Education_PhD                   0.011592
HasCoSigner_Yes                 0.010977
EmploymentType_Unemployed       0.

# Observation on Embedded Feature Selection Methods

Embedded feature selection methods perform feature selection during the model training process itself. Unlike filter methods and wrapper methods, embedded methods integrate feature selection directly into model optimization.

Two embedded methods were applied on the loan default dataset:

1. L1 Regularization (Lasso)
2. Tree-Based Feature Importance using Random Forest

## 1. L1 Regularization (Lasso)

Logistic Regression with L1 regularization was used. L1 regularization adds a penalty term to the loss function which forces weak feature coefficients toward zero.

The model achieved an accuracy of:

* 0.8852

The selected features with larger coefficients included:

* Age
* Income
* LoanAmount
* InterestRate
* MonthsEmployed
* CreditScore

Since most coefficients remained non-zero, the method retained almost all features. This indicates that many variables in the dataset contain useful predictive information.

The parameter `C` controls regularization strength:

* Smaller `C` → stronger regularization → more feature elimination
* Larger `C` → weaker regularization

In this experiment, `C=0.1` was not strong enough to eliminate many features completely.

---

## 2. Tree-Based Feature Importance (Random Forest)

Random Forest feature importance was used to evaluate how much each feature contributed to reducing impurity across decision trees.

The model achieved an accuracy of:

* 0.8853

The most important features identified were:

* Income
* InterestRate
* LoanAmount
* Age
* CreditScore
* MonthsEmployed

Tree-based importance assigns an importance score to every feature rather than removing them directly. Features with very low importance can later be removed using a threshold.

The threshold used was:

* importance > 0.01

Because the threshold was relatively low, most features were retained.

---

## Overall Observation

Across wrapper methods and embedded methods, several features consistently appeared as important predictors:

* Age
* Income
* InterestRate
* LoanAmount
* MonthsEmployed

This consistency across multiple feature selection techniques suggests that these variables are genuinely strong predictors of loan default.

The embedded methods maintained accuracy close to the baseline model while automatically identifying important features during training. Compared to wrapper methods, embedded methods are computationally faster and scale better for larger datasets.
